# CatBoost final: churn con presupuesto secuencial

Notebook reducido al mejor modelo de la segunda iteracion: **CatBoost GPU + calibracion isotonic**. Reproduce el split, entrenamiento OOF, calibracion y regla de intervencion usados para alcanzar al menos **S/27 660**.

## TL;DR

- Split 80/20 estratificado, semilla 42.
- Cinco folds OOF para calibrar sin usar test.
- Intervenir si `p(churn) > 0.10`, mayor probabilidad primero.
- Presupuesto inicial S/1 000; costo S/10; exito suma S/100.
- El notebook falla con un `assert` si baja de S/27 660; cualquier mejora se conserva.

## 1. Configuracion y datos

Se exige GPU Tesla T4. `customerID` solo se conserva para desempatar el ranking; no entra al modelo.

In [ ]:
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
INITIAL_BUDGET = 1000
INTERVENTION_COST = 10
SUCCESS_REWARD = 100
THRESHOLD = INTERVENTION_COST / SUCCESS_REWARD
MINIMUM_FINAL_BUDGET = 27660
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert "T4" in gpu_name, f"Se esperaba Tesla T4; se obtuvo {gpu_name!r}"

data_path = next(Path("/kaggle/input").rglob("telco.csv"))
df = pd.read_csv(data_path)
assert len(df) == 7043 and df["customerID"].is_unique
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")
assert (df["TotalCharges"].isna() == (df["tenure"] == 0)).all()
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)
df["ChurnFlag"] = (df["Churn"] == "Yes").astype(np.int8)

feature_columns = [c for c in df.columns if c not in {"customerID", "Churn", "ChurnFlag"}]
categorical_columns = [c for c in feature_columns if c not in {"tenure", "MonthlyCharges", "TotalCharges"}]
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(
    indices, test_size=0.20, stratify=df["ChurnFlag"], random_state=SEED
)
assert (len(train_idx), len(test_idx)) == (5634, 1409)
assert set(train_idx).isdisjoint(test_idx)

X_train = df.loc[train_idx, feature_columns].reset_index(drop=True)
y_train = df.loc[train_idx, "ChurnFlag"].to_numpy()
X_test = df.loc[test_idx, feature_columns].reset_index(drop=True)
y_test = df.loc[test_idx, "ChurnFlag"].to_numpy()
ids_test = df.loc[test_idx, "customerID"].astype(str).to_numpy()
X_train[categorical_columns] = X_train[categorical_columns].astype(str)
X_test[categorical_columns] = X_test[categorical_columns].astype(str)

print({"gpu": gpu_name, "train": X_train.shape, "test": X_test.shape})

## 2. Predicciones OOF y calibracion

Cada fila de train se predice con un modelo que no la uso para ajustar. Isotonic se entrena sobre esos scores OOF, igual que en la ejecucion completa.

In [ ]:
def new_catboost(iterations=500):
    return CatBoostClassifier(
        iterations=iterations,
        depth=6,
        learning_rate=0.03,
        loss_function="Logloss",
        eval_metric="Logloss",
        task_type="GPU",
        devices="0",
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
    )


oof_scores = np.zeros(len(X_train))
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (fit_idx, valid_idx) in enumerate(folds.split(X_train, y_train), start=1):
    model = new_catboost()
    model.fit(
        X_train.iloc[fit_idx],
        y_train[fit_idx],
        cat_features=categorical_columns,
        eval_set=(X_train.iloc[valid_idx], y_train[valid_idx]),
        early_stopping_rounds=50,
        verbose=False,
    )
    oof_scores[valid_idx] = model.predict_proba(X_train.iloc[valid_idx])[:, 1]
    print(f"fold={fold}/5")

calibrator = IsotonicRegression(out_of_bounds="clip").fit(oof_scores, y_train)
oof_calibrated = calibrator.predict(oof_scores)
print({
    "roc_auc_oof": round(roc_auc_score(y_train, oof_calibrated), 4),
    "pr_auc_oof": round(average_precision_score(y_train, oof_calibrated), 4),
    "brier_oof": round(brier_score_loss(y_train, oof_calibrated), 4),
})

## 3. Modelo final y ranking congelado

El modelo final usa solo train. Probabilidades se calibran antes de seleccionar y ordenar clientes de test.

In [ ]:
final_model = new_catboost()
final_model.fit(X_train, y_train, cat_features=categorical_columns, verbose=False)
test_raw_scores = final_model.predict_proba(X_test)[:, 1]
test_scores = calibrator.predict(test_raw_scores)

eligible = np.flatnonzero(test_scores > THRESHOLD)
ordered_indices = eligible[np.lexsort((ids_test[eligible], -test_scores[eligible]))]
assert (test_scores[ordered_indices] > THRESHOLD).all()
assert np.all(test_scores[ordered_indices][:-1] >= test_scores[ordered_indices][1:])
print({"planned_interventions": len(ordered_indices), "minimum_score": float(test_scores[ordered_indices].min())})

## 4. Simulacion y comprobacion de no regresion

La etiqueta se consulta solo despues de pagar cada intervencion. El orden ya esta congelado.

In [ ]:
budget = INITIAL_BUDGET
ledger_rows = []
for rank, row_index in enumerate(ordered_indices, start=1):
    if budget < INTERVENTION_COST:
        break
    budget_before = budget
    budget -= INTERVENTION_COST
    success = int(y_test[row_index]) == 1
    if success:
        budget += SUCCESS_REWARD
    ledger_rows.append({
        "rank": rank,
        "customerID": ids_test[row_index],
        "calibrated_probability": test_scores[row_index],
        "actual_churn": int(y_test[row_index]),
        "budget_before": budget_before,
        "success": success,
        "budget_after": budget,
        "cumulative_profit": budget - INITIAL_BUDGET,
    })

ledger = pd.DataFrame(ledger_rows)
profit = budget - INITIAL_BUDGET
successes = int(ledger["success"].sum())
assert profit == successes * SUCCESS_REWARD - len(ledger) * INTERVENTION_COST
assert budget >= MINIMUM_FINAL_BUDGET, f"Presupuesto minimo {MINIMUM_FINAL_BUDGET}; obtenido {budget}"

summary = {
    "status": "complete",
    "model": "CatBoost",
    "gpu": gpu_name,
    "seed": SEED,
    "calibration": "isotonic sobre scores OOF",
    "initial_budget": INITIAL_BUDGET,
    "minimum_expected_budget": MINIMUM_FINAL_BUDGET,
    "final_budget": int(budget),
    "profit": int(profit),
    "planned_interventions": int(len(ordered_indices)),
    "executed_interventions": int(len(ledger)),
    "successes": successes,
    "failures": int(len(ledger) - successes),
    "stopped_for_budget": bool(len(ledger) < len(ordered_indices)),
    "roc_auc_test": float(roc_auc_score(y_test, test_scores)),
    "pr_auc_test": float(average_precision_score(y_test, test_scores)),
    "brier_test": float(brier_score_loss(y_test, test_scores)),
}
ledger.to_csv(OUTPUT_DIR / "intervention_ledger.csv", index=False)
(OUTPUT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Takeaways

Resultado aceptado solo cuando la ultima celda termina sin error y confirma presupuesto final de al menos **S/27 660**. Una mejora se conserva. El umbral de 0.10 deriva de `costo/recompensa`; CatBoost usa GPU, mientras calibracion y simulacion son operaciones pequenas de CPU.